# QLoRA Fine-Tuning Project — Banking77

This notebook is the clean **start-to-baseline** version of the project.

## Goal

We will:

1. Load the current Parquet-backed `mteb/banking77` dataset.
2. Keep 10 banking intents so the first experiment stays small.
3. Convert examples into conversational prompt/completion format for SFT.
4. Load `Qwen/Qwen3-1.7B` in 4-bit NF4.
5. Run deterministic baseline inference **before fine-tuning**.
6. Evaluate and save baseline results.

The next notebook stage will attach LoRA adapters and perform SFT.

> In Google Colab, select **Runtime → Change runtime type → GPU** before running the notebook.


## 1. Install dependencies

In [ ]:
!pip install -q -U transformers datasets trl peft accelerate bitsandbytes tqdm

If Colab asks you to restart the runtime after installation, restart it and continue from the imports cell below.

## 2. Imports and environment check

In [ ]:
import json
from pathlib import Path

import torch
import transformers
import datasets
import peft
import trl
import bitsandbytes as bnb

from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("bitsandbytes:", bnb.__version__)

## 3. Verify that Colab has a GPU

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU was detected. In Colab select "
        "Runtime → Change runtime type → GPU, then rerun the notebook."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA version reported by PyTorch:", torch.version.cuda)
print("BF16 supported:", torch.cuda.is_bf16_supported())

## 4. Project constants

In [ ]:
MODEL_ID = "Qwen/Qwen3-1.7B"
DATASET_ID = "mteb/banking77"

SEED = 42
BASELINE_EVAL_SIZE = 100

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL_ID)
print("Dataset:", DATASET_ID)

## 5. Load Banking77

We use `mteb/banking77`, which is stored as Parquet and does not rely on the old Python dataset script that caused the earlier error.

In [ ]:
dataset = load_dataset(DATASET_ID)

print(dataset)
print("\nFirst training example:")
print(dataset["train"][0])

## 6. Inspect labels

In [ ]:
all_labels = sorted(set(dataset["train"]["label_text"]))

print("Number of unique labels:", len(all_labels))
print("\nFirst 20 labels:")
for label in all_labels[:20]:
    print("-", label)

## 7. Select 10 intents

We intentionally start with 10 classes. The goal of this first project is to learn the fine-tuning pipeline, not to maximize benchmark difficulty.

In [ ]:
CHOSEN_LABELS = [
    "activate_my_card",
    "age_limit",
    "atm_support",
    "cancel_transfer",
    "card_arrival",
    "card_not_working",
    "cash_withdrawal_charge",
    "change_pin",
    "compromised_card",
    "cash_withdrawal_not_recognised",
]

missing_labels = sorted(set(CHOSEN_LABELS) - set(all_labels))

if missing_labels:
    raise ValueError(
        f"These requested labels were not found in the dataset: {missing_labels}"
    )

train_ds = dataset["train"].filter(
    lambda example: example["label_text"] in CHOSEN_LABELS
)

test_ds = dataset["test"].filter(
    lambda example: example["label_text"] in CHOSEN_LABELS
)

print("Filtered training examples:", len(train_ds))
print("Filtered test examples:", len(test_ds))

## 8. Create the instruction prompt

In [ ]:
allowed_labels = "\n".join(f"- {label}" for label in CHOSEN_LABELS)

SYSTEM_PROMPT = f"""You are a banking support intent classifier.

Classify the user's request into exactly one of the following labels:

{allowed_labels}

Return ONLY the label. Do not add explanations, punctuation, or extra text."""

print(SYSTEM_PROMPT)

## 9. Convert dataset examples to SFT prompt/completion format

In [ ]:
def to_sft_example(example: dict) -> dict:
    """
    Convert one Banking77 example into conversational prompt/completion format.

    The prompt contains the system instruction and the customer's banking
    query. The completion contains the ground-truth intent label that the
    assistant should generate during supervised fine-tuning.

    Parameters
    ----------
    example : dict
        A Banking77 example containing at least the fields ``text`` and
        ``label_text``.

    Returns
    -------
    dict
        A dictionary with two fields:

        - ``prompt``: system and user chat messages.
        - ``completion``: the expected assistant message containing the
          correct intent label.
    """
    return {
        "prompt": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": example["text"],
            },
        ],
        "completion": [
            {
                "role": "assistant",
                "content": example["label_text"],
            }
        ],
    }


train_sft = train_ds.map(
    to_sft_example,
    remove_columns=train_ds.column_names,
)

test_sft = test_ds.map(
    to_sft_example,
    remove_columns=test_ds.column_names,
)

print(train_sft[0])

At this point we have the format that will later be passed to TRL's `SFTTrainer`. We are **not training yet**.

## 10. Choose the 4-bit compute dtype

In [ ]:
COMPUTE_DTYPE = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

print("4-bit compute dtype:", COMPUTE_DTYPE)

## 11. Load the tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("EOS token:", tokenizer.eos_token)
print("EOS token id:", tokenizer.eos_token_id)
print("Pad token id:", tokenizer.pad_token_id)

## 12. Configure 4-bit NF4 quantization

The base model is loaded in 4-bit to reduce memory usage. Later, QLoRA will keep these base weights frozen and train LoRA adapters.

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

print(quantization_config)

## 13. Load the base model

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype="auto",
)

model.eval()

print("Model loaded.")
print("Primary device:", model.device)

## 14. Define deterministic baseline inference

Qwen3 supports a thinking mode. For this short classification task we explicitly disable it.

We also use greedy decoding (`do_sample=False`) so that the benchmark is deterministic.

In [ ]:
@torch.inference_mode()
def predict_intent(query: str) -> str:
    """
    Predict a Banking77 intent label for one customer query.

    The function formats the query using Qwen3's chat template, explicitly
    disables thinking mode, performs deterministic greedy generation, and
    returns only the newly generated text.

    Parameters
    ----------
    query : str
        A banking-related customer request to classify.

    Returns
    -------
    str
        The raw intent text generated by the current model.
    """
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": query,
        },
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    generated_tokens = outputs[0, inputs["input_ids"].shape[1]:]

    prediction = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
    )

    return prediction.strip()

## 15. Sanity-check a few manual queries

In [ ]:
manual_queries = [
    "My card still hasn't arrived.",
    "How do I change my PIN?",
    "Someone got access to my card details.",
    "Why was I charged for withdrawing cash?",
    "I want to cancel a transfer.",
]

for query in manual_queries:
    print("Query:     ", query)
    print("Prediction:", predict_intent(query))
    print("-" * 70)

## 16. Create a fixed baseline evaluation subset

In [ ]:
baseline_eval_ds = test_ds.shuffle(seed=SEED).select(
    range(min(BASELINE_EVAL_SIZE, len(test_ds)))
)

print("Baseline evaluation examples:", len(baseline_eval_ds))
print(baseline_eval_ds[0])

## 17. Label normalization helper

In [ ]:
def normalize_label(text: str) -> str:
    """
    Normalize a model-generated label for exact-match evaluation.

    The function lowercases the output, keeps only the first non-empty line,
    and removes harmless surrounding whitespace and quoting characters. It
    does not try to infer a label from a long explanation because the model
    was explicitly instructed to return only one label.

    Parameters
    ----------
    text : str
        Raw generated model output.

    Returns
    -------
    str
        Normalized label text used for exact-match comparison.
    """
    normalized = text.strip().lower()

    non_empty_lines = [
        line.strip()
        for line in normalized.splitlines()
        if line.strip()
    ]

    if not non_empty_lines:
        return ""

    normalized = non_empty_lines[0]
    normalized = normalized.strip("`'\" ")

    return normalized

## 18. Baseline evaluation function

In [ ]:
def evaluate_model(eval_dataset) -> dict:
    """
    Evaluate the current model using exact-match intent classification accuracy.

    Each query is sent through ``predict_intent``. The normalized generated
    label is compared with the normalized ground-truth ``label_text`` value.

    Parameters
    ----------
    eval_dataset : datasets.Dataset
        A dataset containing the columns ``text`` and ``label_text``.

    Returns
    -------
    dict
        Evaluation results containing:

        - ``accuracy``: fraction of predictions that exactly match the target.
        - ``correct``: number of correct predictions.
        - ``total``: number of evaluated examples.
        - ``predictions``: per-example query, target, prediction, and result.
    """
    predictions = []
    correct = 0

    for example in tqdm(eval_dataset):
        query = example["text"]
        expected = normalize_label(example["label_text"])
        predicted = normalize_label(predict_intent(query))

        is_correct = predicted == expected
        correct += int(is_correct)

        predictions.append(
            {
                "query": query,
                "expected": expected,
                "predicted": predicted,
                "correct": is_correct,
            }
        )

    total = len(eval_dataset)
    accuracy = correct / total if total else 0.0

    return {
        "accuracy": accuracy,
        "correct": correct,
        "total": total,
        "predictions": predictions,
    }

## 19. Run the baseline evaluation

In [ ]:
baseline_results = evaluate_model(baseline_eval_ds)

print(
    f"Baseline accuracy: "
    f"{baseline_results['correct']}/{baseline_results['total']} "
    f"= {baseline_results['accuracy']:.2%}"
)

## 20. Inspect baseline mistakes

In [ ]:
baseline_mistakes = [
    result
    for result in baseline_results["predictions"]
    if not result["correct"]
]

print("Number of mistakes:", len(baseline_mistakes))

for mistake in baseline_mistakes[:10]:
    print("Query:    ", mistake["query"])
    print("Expected: ", mistake["expected"])
    print("Predicted:", mistake["predicted"])
    print("-" * 70)

## 21. Save baseline results

In [ ]:
baseline_path = RESULTS_DIR / "baseline.json"

with baseline_path.open("w", encoding="utf-8") as file:
    json.dump(
        baseline_results,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("Saved:", baseline_path.resolve())

## 22. Stop here

At this point you should have:

- a working dataset,
- a 10-class SFT dataset,
- a 4-bit Qwen3-1.7B base model,
- deterministic inference,
- a fixed baseline evaluation set,
- `results/baseline.json`.

**Do not attach LoRA until this notebook runs cleanly through this point.**

The next stage will:

1. prepare the 4-bit model for k-bit training,
2. attach LoRA adapters,
3. inspect trainable parameter counts,
4. configure `SFTTrainer`,
5. train,
6. evaluate on the exact same baseline subset,
7. compare before vs. after.

### Current references

- Qwen3-1.7B model card: https://huggingface.co/Qwen/Qwen3-1.7B
- Banking77 Parquet dataset: https://huggingface.co/datasets/mteb/banking77
- Transformers bitsandbytes guide: https://huggingface.co/docs/transformers/main/quantization/bitsandbytes
- TRL SFTTrainer: https://huggingface.co/docs/trl/main/sft_trainer
